# Lab 3: Build a Search Agent

In this lab, we'll use the Microsoft Foundry Agent Service to create an agent that is able to retrieve information from documents stored in Azure AI Search, a vector database. This pattern is known as retrieval augmented generation or RAG. The documents that we'll be searching are health insurance policies.

#### Step 1: Load packages

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, AzureAISearchTool, AzureAISearchToolResource, AISearchIndexResource
from azure.core.exceptions import HttpResponseError
from azure.identity import DefaultAzureCredential

load_dotenv()

#### Step 2: Connect to your Microsoft Foundry Project

Use a token credential for project and agent operations. The code uses `AzureCliCredential` first and then `DefaultAzureCredential` as a fallback.

In [ ]:
# Connecting to our Microsoft Foundry project, which will allow us to use the deployed gpt-5.4 model
project = AIProjectClient(
    endpoint=os.getenv("AIPROJECT_ENDPOINT"),
    credential=DefaultAzureCredential()
)

#### Step 3: Connect to your Azure AI Search index

In [ ]:
# First enter the name of your search index

index_name="health-plan"
print(index_name)

This code retrieves the connection ID for your Azure AI Search resource, then defines and configures the search tool and its resources. It ensures your agent is set up to access the correct search index, enabling it to perform document retrieval using the Microsoft Foundry Agent Service SDK.

In [ ]:
# Iterate through the connections in your project and get the connection ID of the Azure AI Search connection.
conn_id = None
for conn in project.connections.list():
    if getattr(conn, "type", None) == "CognitiveSearch":
        conn_id = conn.id
        break

if not conn_id:
    raise ValueError("No Azure Cognitive Search connection found in this project.")

# Build the Azure AI Search tool using azure.ai.projects.models classes
# AISearchIndexResource uses project_connection_id and index_name
# AzureAISearchToolResource uses indexes (list)
# AzureAISearchTool wraps the resource
ai_search_tool = AzureAISearchTool(
    azure_ai_search=AzureAISearchToolResource(
        indexes=[
            AISearchIndexResource(
                project_connection_id=conn_id,
                index_name=index_name,
            )
        ]
    )
)
print(f"Configured AI Search tool for index: {index_name}")

#### Step 4: Define the search agent

In this step, you will define and create the search agent using the Microsoft Foundry Agent Service SDK. The agent is configured with the GPT-5.4 model, a descriptive name, instructions for its behavior, and the search tool and resources you set up previously. This setup enables the agent to process user queries and retrieve relevant information from your Azure AI Search index, making it capable of intelligent, document-grounded search.

In [ ]:
agent_name = "search-agent"
model_name = os.environ.get("CHAT_MODEL", "gpt-5.4")
instructions = "You are a helpful agent that is an expert at searching health plan documents."

# Create the agent with the AI Search tool attached
agent_definition = PromptAgentDefinition(
    model=model_name,
    instructions=instructions,
    tools=[ai_search_tool],
)

search_agent = project.agents.create_version(
    agent_name=agent_name,
    definition=agent_definition,
)
print(f"Created search agent version, ID: {search_agent.id}")

#### Step 5: Chat with the search agent

In this step, you'll interact with the search agent by sending it a natural language query and viewing its response. The code demonstrates how to:

- Create a conversation for interacting with the agent.
- Send a user query to the agent using the Azure AI Foundry SDK.
- Receive and display the agent's response based on the configured search knowledge source.
- Clean up by deleting the agent after the interaction is complete.

This exercise demonstrates how a search-enabled AI agent can use retrieval-augmented generation (RAG) to answer questions using information from indexed health plan documents.

In [ ]:
# The name of the health plan we want to search for
plan_name = 'Northwind Standard'

# Create a conversation and send the user query
openai = project.get_openai_client(agent_name=agent_name)
conversation = openai.conversations.create()
print(f"Created conversation, ID: {conversation.id}")

response = openai.responses.create(
    conversation=conversation.id,
    input=f"Tell me about the {plan_name} plan.",
)
print(f"Run finished with status: completed")
print(f"\nAgent: {response.output_text}")

# Delete the agent when done
project.agents.delete_version(agent_name=agent_name, agent_version=search_agent.version)
print("Deleted agent")